# **Task 1 — Getting and Cleaning the Data**  
### Data Science Capstone — SwiftKey Project  
**Author:** Sonal Kumari

This notebook performs all steps required for **Task 1**:
1. Loading and sampling the large HC Corpora dataset  
2. Cleaning text (lowercasing, removing URLs, numbers, extra spaces, non-ASCII)  
3. Tokenization  
4. Profanity filtering  
5. Saving cleaned & tokenized datasets  


In [1]:
import re
import random
from pathlib import Path
from collections import Counter

# === EDIT THIS path to your dataset location ===
dataset_root = Path(r"C:\Users\Lenovo\OneDrive\Desktop\swiftkey-capstone\final\en_US")

blogs_path = dataset_root / "en_US.blogs.txt"
news_path = dataset_root / "en_US.news.txt"
twitter_path = dataset_root / "en_US.twitter.txt"

# Output folders (created automatically)
project_root = Path(".")
sampled_dir = project_root / "data" / "sampled"
cleaned_dir = project_root / "data" / "cleaned"
tokenized_dir = project_root / "data" / "tokenized"

for p in [sampled_dir, cleaned_dir, tokenized_dir]:
    p.mkdir(parents=True, exist_ok=True)

print("Paths loaded successfully!")


Paths loaded successfully!


### 📌 Why Sampling?

The Blogs/News/Twitter files are very large (hundreds of MB).  
Working on the full dataset is slow, so we create a **representative sample** (~0.5%).

This improves speed without reducing quality significantly.


In [3]:
def sample_file(src_path, dest_path, sample_fraction=0.005, seed=42):
    random.seed(seed)
    total, kept = 0, 0

    with open(src_path, "r", encoding="utf-8", errors="ignore") as src, \
         open(dest_path, "w", encoding="utf-8") as dst:

        for line in src:
            total += 1
            if random.random() <= sample_fraction:
                dst.write(line)
                kept += 1

    print(f"Sampled {kept}/{total} lines → {dest_path.name}")

# === Run Sampling ===
sample_file(blogs_path, sampled_dir/"blogs_sample.txt")
sample_file(news_path, sampled_dir/"news_sample.txt")
sample_file(twitter_path, sampled_dir/"twitter_sample.txt")


Sampled 4496/899288 lines → blogs_sample.txt
Sampled 5026/1010242 lines → news_sample.txt
Sampled 11821/2360148 lines → twitter_sample.txt


### 📌 Cleaning Includes:
- Lowercasing  
- Removing URLs  
- Removing numbers  
- Removing non-ASCII  
- Removing extra spaces  

This prepares the text for tokenization.


In [4]:
URL_RE = re.compile(r'https?://\S+|www\.\S+')
NUM_RE = re.compile(r'\b\d+\b')
WS_RE = re.compile(r'\s+')
NON_ASCII_RE = re.compile(r'[^\x00-\x7F]+')

def clean_text(text):
    text = text.lower()
    text = URL_RE.sub(" ", text)
    text = NUM_RE.sub(" ", text)
    text = NON_ASCII_RE.sub(" ", text)
    text = WS_RE.sub(" ", text).strip()
    return text

def clean_file(in_path, out_path):
    with open(in_path, "r", encoding="utf-8", errors="ignore") as src, \
         open(out_path, "w", encoding="utf-8") as dst:

        for line in src:
            cleaned = clean_text(line)
            if cleaned:
                dst.write(cleaned + "\n")

    print(f"Cleaned file → {out_path.name}")

# === Run Cleaning ===
clean_file(sampled_dir/"blogs_sample.txt", cleaned_dir/"blogs_clean.txt")
clean_file(sampled_dir/"news_sample.txt", cleaned_dir/"news_clean.txt")
clean_file(sampled_dir/"twitter_sample.txt", cleaned_dir/"twitter_clean.txt")


Cleaned file → blogs_clean.txt
Cleaned file → news_clean.txt
Cleaned file → twitter_clean.txt


### 📌 Why Profanity Filtering?

The dataset contains offensive words.  
We remove lines containing profanity so the model does not learn inappropriate predictions.


In [5]:
# Small built-in profanity list (you can expand it)
profanity_list = {
    "fuck","shit","bitch","asshole","slut","whore","bastard","crap","damn"
}

def line_contains_profanity(line):
    words = re.findall(r"\b[\w']+\b", line.lower())
    return any(w in profanity_list for w in words)

def remove_profanity(in_path, out_path):
    kept = 0
    total = 0

    with open(in_path, "r", encoding="utf-8") as src, \
         open(out_path, "w", encoding="utf-8") as dst:

        for line in src:
            total += 1
            if not line_contains_profanity(line):
                dst.write(line)
                kept += 1

    print(f"Removed profanity: Kept {kept}/{total} lines → {out_path.name}")

# === Apply to cleaned files ===
remove_profanity(cleaned_dir/"blogs_clean.txt", cleaned_dir/"blogs_clean_noprofanity.txt")
remove_profanity(cleaned_dir/"news_clean.txt", cleaned_dir/"news_clean_noprofanity.txt")
remove_profanity(cleaned_dir/"twitter_clean.txt", cleaned_dir/"twitter_clean_noprofanity.txt")


Removed profanity: Kept 4454/4494 lines → blogs_clean_noprofanity.txt
Removed profanity: Kept 5021/5024 lines → news_clean_noprofanity.txt
Removed profanity: Kept 11565/11821 lines → twitter_clean_noprofanity.txt


### 📌 Tokenization:
This splits text into individual **tokens** (words).  
We will use a simple regex-based tokenizer.


In [6]:
WORD_RE = re.compile(r"\b[\w']+\b")

def tokenize_line(line):
    return WORD_RE.findall(line.lower())

def tokenize_file(in_path, out_path):
    with open(in_path, "r", encoding="utf-8") as src, \
         open(out_path, "w", encoding="utf-8") as dst:

        for line in src:
            tokens = tokenize_line(line)
            if tokens:
                dst.write(" ".join(tokens) + "\n")

    print(f"Tokenized → {out_path.name}")

# Run tokenization
tokenize_file(cleaned_dir/"blogs_clean_noprofanity.txt", tokenized_dir/"blogs_tokenized.txt")
tokenize_file(cleaned_dir/"news_clean_noprofanity.txt", tokenized_dir/"news_tokenized.txt")
tokenize_file(cleaned_dir/"twitter_clean_noprofanity.txt", tokenized_dir/"twitter_tokenized.txt")


Tokenized → blogs_tokenized.txt
Tokenized → news_tokenized.txt
Tokenized → twitter_tokenized.txt


### 📊 Basic Token Statistics
Let's inspect the top 20 most frequent words in each dataset.


In [8]:
def top_tokens(path, n=20):
    c = Counter()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            c.update(line.split())
    return c.most_common(n)

print("Blogs:", top_tokens(tokenized_dir/"blogs_tokenized.txt"))
print("News:", top_tokens(tokenized_dir/"news_tokenized.txt"))
print("Twitter:", top_tokens(tokenized_dir/"twitter_tokenized.txt"))


Blogs: [('the', 9278), ('to', 5347), ('and', 5263), ('a', 4371), ('of', 4326), ('i', 4096), ('in', 2991), ('that', 2380), ('it', 2227), ('is', 2141), ('for', 1736), ('you', 1735), ('my', 1430), ('with', 1365), ('was', 1326), ('on', 1313), ('this', 1287), ('as', 1156), ('have', 1128), ('be', 1076)]
News: [('the', 9778), ('to', 4600), ('a', 4490), ('and', 4390), ('of', 3940), ('in', 3431), ('for', 1808), ('that', 1749), ('is', 1480), ('on', 1295), ('with', 1242), ('it', 1235), ('said', 1205), ('was', 1126), ('he', 1117), ('at', 1096), ('i', 912), ('as', 894), ('but', 775), ('his', 769)]
Twitter: [('the', 4546), ('to', 3902), ('i', 3619), ('a', 2990), ('you', 2594), ('and', 2162), ('for', 1935), ('in', 1865), ('is', 1767), ('of', 1758), ('it', 1506), ('my', 1453), ('on', 1378), ('that', 1178), ('me', 1011), ('at', 970), ('be', 937), ('with', 882), ('have', 845), ('your', 799)]


This notebook performed:
✔ Sampling  
✔ Cleaning  
✔ Profanity removal  
✔ Tokenization  
✔ Basic statistics  

Next Step → **Task 2: Exploratory Data Analysis (EDA)**  